# 🇮🇩 NLP Bahasa Indonesia dengan IndoBERT

Pakai **IndoBERT** (BERT yang dilatih khusus bahasa Indonesia) untuk mengubah
kalimat menjadi *embedding* — lalu bangun **pendeteksi kemiripan** dan
**klasifikasi sentimen** sederhana di atasnya. Model diambil dari folder bersama
server (tanpa download ±500 MB!).

**Yang kamu pelajari:** tokenisasi, mean-pooling embedding, cosine similarity,
dan melatih classifier ringan di atas fitur BERT.

In [ ]:
# Sel 1 — muat IndoBERT dari model bersama server (fallback: download).
import os, torch
from transformers import AutoTokenizer, AutoModel

sumber = "/opt/ch-models/indobert-base-p1"
if not os.path.isdir(sumber):
    sumber = "indobenchmark/indobert-base-p1"
tok = AutoTokenizer.from_pretrained(sumber)
bert = AutoModel.from_pretrained(sumber).cuda().eval()
print("IndoBERT siap dari:", sumber)

In [ ]:
# Sel 2 — fungsi embedding kalimat (mean pooling atas token).
@torch.no_grad()
def embed(kalimat: list[str]) -> torch.Tensor:
    enc = tok(kalimat, padding=True, truncation=True, max_length=64, return_tensors="pt")
    enc = {k: v.cuda() for k, v in enc.items()}
    out = bert(**enc).last_hidden_state          # [batch, token, 768]
    mask = enc["attention_mask"].unsqueeze(-1)
    vec = (out * mask).sum(1) / mask.sum(1)       # rata-rata token nyata
    return torch.nn.functional.normalize(vec, dim=1).cpu()

emb = embed(["kuliah pagi ini seru sekali", "dosen menjelaskan dengan menarik",
             "server gpu kampus sangat membantu", "makanan di kantin kurang enak"])
print("Bentuk embedding:", tuple(emb.shape))

In [ ]:
# Sel 3 — kemiripan antar kalimat (cosine similarity) + heatmap.
import matplotlib.pyplot as plt

sim = emb @ emb.T
label = ["kuliah seru", "dosen menarik", "GPU membantu", "kantin kurang"]
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(sim, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(4), label, rotation=30, ha="right")
ax.set_yticks(range(4), label)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", color="white")
fig.colorbar(im); ax.set_title("Kemiripan semantik (IndoBERT)"); plt.tight_layout()

In [ ]:
# Sel 4 — klasifikasi sentimen mini: fitur BERT + LogisticRegression.
from sklearn.linear_model import LogisticRegression

data = [
    ("aplikasinya bagus dan cepat", 1), ("pelayanan ramah, sangat puas", 1),
    ("fiturnya lengkap, mantap", 1),    ("keren banget, recommended", 1),
    ("lambat dan sering error", 0),     ("kecewa, tidak sesuai harapan", 0),
    ("jelek, buang-buang uang", 0),     ("parah, tidak bisa dipakai", 0),
]
teks, y = [t for t, _ in data], [l for _, l in data]
clf = LogisticRegression().fit(embed(teks), y)

uji = ["produk ini luar biasa membantu", "menyesal beli, kualitas buruk"]
for kalimat, prob in zip(uji, clf.predict_proba(embed(uji))[:, 1]):
    print(f"{prob:.0%} positif — {kalimat}")

**Langkah lanjut:** ganti data mini dengan dataset asli (upload CSV → `pandas`),
atau **fine-tune** penuh dengan `transformers.Trainer` untuk akurasi maksimal.
Dataset sentimen Indonesia populer: IndoNLU (SmSA).